In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve, average_precision_score
import pandas as pd
import numpy as np
import os, math

os.chdir('../../../results/PlantCAD2_tasks/')

In [ ]:
species  = ['arabidopsis_thaliana', 'eutrema_salsugineum',
            'populus_trichocarpa', 'phaseolus_vulgaris', 
             'glycine_max', 'brachypodium_distachyon', 
           'hordeum_vulgare', 'oryza_sativa', 'setaria_viridis',
           'sorghum_bicolor',  'zea_mays']
contexts = ['c300', 'c600', 'c1000']
ckpts = [
    # 'pcv2-l24-d0768-checkpoints-lr-1e-4',
    # 'pcv2-l48-d1024-checkpoints-lr-1e-4',
    'pcv2-l48-d1536-checkpoints-lr-1e-4',
    'agront-checkpoints-lr-1e-4',
    'sup_pcv2-l24-d0768-checkpoints-lr-1e-4',
    'cnn-lstm-imbalance-lr-1e-3'
]
pcv2_alphas = [0.4, 0.7, 1.0]

In [ ]:
records = []
for sp in species:
    for ctx in contexts:
        if sp == 'arabidopsis_thaliana':
            labels_path = f'accessible_angiosperm_{ctx}/data/valid.tsv'
        else:
            labels_path = f'accessible_angiosperm_{ctx}/data/test_{sp}.tsv'
        labs = pd.read_csv(labels_path, sep='\t')
        y_true = labs['Label']

        for k, ckpt in enumerate(ckpts):
            print(f"[{sp} | {ctx}] Processing model {k+1}/{len(ckpts)}: {ckpt}")
            if sp == 'arabidopsis_thaliana':
                scores_path = f'accessible_angiosperm_{ctx}/models/{ckpt}/valid_scores.tsv'
            else:
                scores_path = f'accessible_angiosperm_{ctx}/models/{ckpt}/test_{sp}_scores.tsv'
            preds = pd.read_csv(scores_path, sep='\t')
            y_scores = preds['probability_positive']
            ap = average_precision_score(y_true, y_scores)

            if 'agront' in ckpt.lower():
                color, alpha = 'gray', 1.0
            elif 'sup_pcv2' in ckpt.lower():
                color, alpha = 'C1', 1.0
            elif 'cnn' in ckpt.lower():
                color, alpha = 'C1', 0.5
            else:
                color, alpha = 'C0', pcv2_alphas[k]

            if 'cnn' in ckpt.lower():
                lab = ckpt.replace('-imbalance-lr-1e-3', '')
            else:
                lab = ckpt.replace('-checkpoints-lr-1e-4', '')

            records.append({
                'species': sp,
                'context': ctx,
                'model': lab,
                'ap': ap,
                'color': color,
                'alpha': alpha,
                'positive': y_true.sum(),
                'negative': len(y_true)-y_true.sum(),
                'baseline': y_true.mean()
            })

df = pd.DataFrame(records)

In [ ]:
df.head()

In [ ]:
df.to_csv('../../pipelines/lu_2019_ATACseq/figures/AUPRC.tsv', sep='\t', index=False)

In [ ]:
df = pd.read_csv('../../pipelines/lu_2019_ATACseq/figures/AUPRC.tsv', sep='\t')

In [ ]:
df.head()

In [ ]:
n_sp = len(species)
ncols = 3 
nrows = math.ceil(n_sp / ncols)  

fig, axes = plt.subplots(nrows=nrows, ncols=ncols,
                         figsize=(5 * ncols, 4 * nrows),
                         sharex=True, sharey=True)

axes = np.array(axes).reshape(nrows, ncols)

bar_w = 0.8 / len(ckpts)
x = np.arange(len(contexts))

for idx, sp in enumerate(species):
    row = idx // ncols
    col = idx % ncols
    ax = axes[row, col]

    df_sp = df[df['species'] == sp]

    for i, lab in enumerate(df_sp['model'].unique()):
        sub = df_sp[df_sp['model'] == lab].set_index('context').reindex(contexts).reset_index()
        ax.bar(
            x + i * bar_w,
            sub['ap'],
            bar_w,
            label=f"{lab}",
            color=sub['color'].iloc[0],
            alpha=sub['alpha'].iloc[0]
        )

    for j, ctx in enumerate(contexts):
        baseline = df_sp[df_sp['context'] == ctx]['baseline'].iloc[0]
        ax.hlines(baseline,
                  x[j] - bar_w,
                  x[j] + bar_w * len(ckpts),
                  colors='black', linestyles='--', linewidth=1)

    ax.set_title(sp, fontsize=14, fontweight='bold')
    ax.set_ylabel('AUPRC', fontsize=14, fontweight='bold')
    ax.tick_params(axis='y', labelleft=True)

    tick_pos = x + bar_w * (len(ckpts) - 1) / 2
    ax.set_xticks(tick_pos)
    ax.set_xticklabels(contexts, fontsize=10, rotation=0, ha='center', fontweight='bold')
    ax.tick_params(axis='x', which='both', labelbottom=True)

    ax.set_ylim(0, 1.0)
    ax.legend(loc='upper right', fontsize=7, ncol=2)

for i in range(n_sp, nrows * ncols):
    fig.delaxes(axes.flatten()[i])

plt.tight_layout()
plt.subplots_adjust(hspace=0.25, wspace=0.2)
plt.savefig('../../pipelines/lu_2019_ATACseq/figures/aupr_barplots.pdf', format='pdf', bbox_inches='tight')
plt.show()

In [ ]:
df['species'].unique()

In [ ]:
def plot_bar_by_context(df, contexts, species, output_dir, ckpts):
    os.makedirs(output_dir, exist_ok=True)

    # Example divergence times in million years (update as needed)
    species_divergence = {
        "arabidopsis_thaliana": 0,
        "eutrema_salsugineum": 14,
        "populus_trichocarpa": 100,
        "phaseolus_vulgaris": 120,
        "glycine_max": 120,
        "brachypodium_distachyon": 130,
        "hordeum_vulgare": 135,
        "oryza_sativa": 145,
        "setaria_viridis": 150,
        "sorghum_bicolor": 150,
        "zea_mays": 160,
    }

    models = df['model'].unique()
    n_models = len(models)

    # Filter and order species by divergence time if available
    species_order = sorted(
        [sp for sp in species if sp in df['species'].unique()],
        key=lambda s: species_divergence.get(s, float('inf'))
    )
    n_sp = len(species_order)

    bar_w = 0.8 / n_models
    x = np.arange(n_sp)

    for ctx in contexts:
        df_ctx = df[df['context'] == ctx]
        if df_ctx.empty:
            continue

        fig, ax = plt.subplots(figsize=(5 + 0.5 * n_sp, 6))

        for i, model in enumerate(models):
            sub = df_ctx[df_ctx['model'] == model]
            sub = sub.set_index('species').reindex(species_order).reset_index()

            ax.bar(
                x + i * bar_w,
                sub['ap'],
                bar_w,
                label=model,
                color=sub['color'].iloc[0] if not sub.empty else 'gray',
                alpha=sub['alpha'].iloc[0] if not sub.empty else 0.5
            )

        for j, sp in enumerate(species_order):
            sub = df_ctx[df_ctx['species'] == sp]
            if not sub.empty:
                baseline = sub['baseline'].iloc[0]
                ax.hlines(
                    baseline,
                    x[j] - bar_w,
                    x[j] + bar_w * n_models,
                    colors='black',
                    linestyles='--',
                    linewidth=1
                )

        # Add divergence time to x-tick labels
        xtick_labels = [
            f"{sp}\n({species_divergence[sp]} Mya)" if sp in species_divergence else sp
            for sp in species_order
        ]
        ax.set_xticks(x + bar_w * (n_models - 1) / 2)
        ax.set_xticklabels(xtick_labels, rotation=45, ha='right', fontsize=11, fontweight='bold')

        ax.set_ylabel('AUPRC', fontsize=14, fontweight='bold')
        ax.set_title(f'Model Performance on {ctx} Context Window', fontsize=16, fontweight='bold')
        ax.set_ylim(0, 1.0)
        ax.legend(title='Model', fontsize=10)

        plt.tight_layout()
        outfile = os.path.join(output_dir, f'aupr_barplot_{ctx}.pdf')
        plt.savefig(outfile, format='pdf', bbox_inches='tight')
        plt.close()

In [ ]:
plot_bar_by_context(
    df=df,
    contexts=['c300', 'c600', 'c1000'],  
    species=species,
    output_dir='../../pipelines/lu_2019_ATACseq/figures/',
    ckpts=ckpts
)